# 06 — Plugin Authoring

COGANT is extensible via a small set of `Plugin` protocols under `cogant.plugins.base`. Third-party packages register plugins through the `cogant.plugins` entry-point group in their `pyproject.toml`; COGANT discovers them at runtime with `PluginRegistry`.

This notebook walks through:
1. Inspecting the discovery mechanism with `PluginRegistry`.
2. Writing a minimal in-process `LanguagePlugin` subclass.
3. Registering it into a `PluginRegistry` instance for use in the current process.
4. Listing discovered and loaded plugins.
5. What a packaged plugin's `pyproject.toml` entry-point stanza looks like.

Run from the `cogant/` directory:
```bash
uv run jupyter nbconvert --to notebook --execute docs/notebooks/06_plugin_authoring.ipynb
```

## 1. The discovery mechanism

`PluginRegistry.discover()` enumerates every entry point in the `cogant.plugins` group. On a clean COGANT checkout with no third-party plugins installed, this list is usually empty — which is exactly what we expect and want to verify.

In [1]:
from __future__ import annotations

from cogant.plugins import PluginRegistry, PluginInfo
from cogant.plugins.base import (
    LanguagePlugin,
    PluginMetadata,
)

registry = PluginRegistry()
discovered = registry.discover()
print(f"ENTRY_POINT_GROUP: cogant.plugins")
print(f"Discovered {len(discovered)} plugin(s) via entry points.")
for info in discovered:
    print(f"  - {info.name} ({info.version}) → {info.entry_point}")
if not discovered:
    print("  (none installed — this is normal on a clean checkout)")

ENTRY_POINT_GROUP: cogant.plugins
Discovered 0 plugin(s) via entry points.
  (none installed — this is normal on a clean checkout)


## 2. Write a minimal in-process `LanguagePlugin`

The `LanguagePlugin` protocol requires four abstract methods: `parse`, `extract_symbols`, `extract_types`, and `resolve_imports`, plus the `initialize` and `shutdown` lifecycle methods inherited from `Plugin`.

This demo plugin pretends to handle an imaginary language called **Toy** whose source code is a flat list of `KEYWORD: name` lines.

In [2]:
from typing import Any

class ToyLanguagePlugin(LanguagePlugin):
    """Tiny LanguagePlugin that parses `KEYWORD: name` lines."""

    supported_languages = {"toy"}

    def __init__(self) -> None:
        super().__init__(
            metadata=PluginMetadata(
                name="toy_language",
                version="0.1.0",
                author="COGANT tutorials",
                description="Demo plugin for the notebook tutorial.",
            )
        )
        self._config: dict[str, Any] = {}

    def initialize(self, config: dict[str, Any]) -> None:
        self._config = dict(config)

    def shutdown(self) -> None:
        self._config.clear()

    def parse(self, source_code: str) -> dict[str, Any]:
        nodes: list[dict[str, Any]] = []
        for lineno, line in enumerate(source_code.splitlines(), start=1):
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            if ":" not in stripped:
                continue
            kw, _, name = stripped.partition(":")
            nodes.append({"kind": kw.strip().lower(), "name": name.strip(), "lineno": lineno})
        return {"language": "toy", "nodes": nodes}

    def extract_symbols(self, ast: dict[str, Any]) -> list[dict[str, Any]]:
        return [{"name": n["name"], "kind": n["kind"]} for n in ast.get("nodes", [])]

    def extract_types(self, ast: dict[str, Any]) -> dict[str, Any]:
        # Toy has no types; return an empty dict to satisfy the contract.
        return {}

    def resolve_imports(self, ast: dict[str, Any]) -> list[str]:
        return [n["name"] for n in ast.get("nodes", []) if n["kind"] == "import"]

toy = ToyLanguagePlugin()
print(f"Instantiated plugin: {toy.metadata.name} v{toy.metadata.version}")

Instantiated plugin: toy_language v0.1.0


### Smoke-test the plugin

Before registering we confirm the plugin actually does something useful on a tiny snippet of toy source.

In [3]:
toy.initialize({"strict": True})

source = """
# a tiny toy program
IMPORT: stdlib
CLASS:  Widget
METHOD: update
"""
ast = toy.parse(source)
print("AST      :", ast)
print("Symbols  :", toy.extract_symbols(ast))
print("Types    :", toy.extract_types(ast))
print("Imports  :", toy.resolve_imports(ast))

AST      : {'language': 'toy', 'nodes': [{'kind': 'import', 'name': 'stdlib', 'lineno': 3}, {'kind': 'class', 'name': 'Widget', 'lineno': 4}, {'kind': 'method', 'name': 'update', 'lineno': 5}]}
Symbols  : [{'name': 'stdlib', 'kind': 'import'}, {'name': 'Widget', 'kind': 'class'}, {'name': 'update', 'kind': 'method'}]
Types    : {}
Imports  : ['stdlib']


## 3. Register the plugin into a `PluginRegistry`

`PluginRegistry` is designed to load plugins from Python entry points, but the cache it maintains is public enough for us to inject an in-process instance for testing or notebooks. We add a `PluginInfo` entry and a loaded object so the rest of the registry API treats our plugin as if it had been discovered.

In [4]:
def register_in_process(registry: PluginRegistry, name: str, plugin_obj: Any) -> PluginInfo:
    """Inject an already-instantiated plugin into a PluginRegistry cache.

    This is the notebook-friendly equivalent of the entry-point loading
    path: it makes *plugin_obj* visible to ``list_plugins``, ``get_plugin_info``,
    and ``get_loaded_object`` without requiring a package install.
    """
    info = PluginInfo(
        name=name,
        version=getattr(getattr(plugin_obj, "metadata", None), "version", "unknown"),
        entry_point=f"<in-process>:{type(plugin_obj).__name__}",
        loaded=True,
        error=None,
    )
    registry._cache[name] = info             # noqa: SLF001 — documented notebook pattern
    registry._loaded_objects[name] = plugin_obj  # noqa: SLF001
    return info

info = register_in_process(registry, "toy_language", toy)
print("Registered:", info)

Registered: PluginInfo(name='toy_language', version='0.1.0', entry_point='<in-process>:ToyLanguagePlugin', loaded=True, error=None)


## 4. Use the registry: list, inspect, retrieve

Once registered, the plugin is visible to every read-only method on the registry.

In [5]:
print("list_plugins() :", registry.list_plugins())
print("get_plugin_info:", registry.get_plugin_info("toy_language"))
loaded_obj = registry.get_loaded_object("toy_language")
print("get_loaded_object:", loaded_obj)
print("Is our instance  :", loaded_obj is toy)

assert registry.list_plugins() == ["toy_language"]
assert loaded_obj is toy
print("\nOK — registry round-trips the plugin object.")

list_plugins() : ['toy_language']
get_plugin_info: PluginInfo(name='toy_language', version='0.1.0', entry_point='<in-process>:ToyLanguagePlugin', loaded=True, error=None)
get_loaded_object: <__main__.ToyLanguagePlugin object at 0x1080a0b30>
Is our instance  : True

OK — registry round-trips the plugin object.


## 5. How a packaged plugin declares itself

The in-process pattern above is fine for tests and notebooks, but a real distribution of the Toy plugin would declare an entry point in its `pyproject.toml`. COGANT will then discover it automatically whenever the package is installed in the active environment.

In [6]:
PYPROJECT_SNIPPET = '''
[project]
name = "cogant-toy-plugin"
version = "0.1.0"
description = "COGANT language plugin for the Toy DSL."
requires-python = ">=3.10"
dependencies = ["cogant>=0.2.0"]

[project.entry-points."cogant.plugins"]
toy_language = "cogant_toy_plugin.plugin:ToyLanguagePlugin"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"
'''
print(PYPROJECT_SNIPPET.strip())

[project]
name = "cogant-toy-plugin"
version = "0.1.0"
description = "COGANT language plugin for the Toy DSL."
requires-python = ">=3.10"
dependencies = ["cogant>=0.2.0"]

[project.entry-points."cogant.plugins"]
toy_language = "cogant_toy_plugin.plugin:ToyLanguagePlugin"

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"


### Discovery flow with a real install

1. User runs `uv pip install cogant-toy-plugin`.
2. `importlib.metadata.entry_points(group="cogant.plugins")` now yields the `toy_language` entry point.
3. `PluginRegistry().discover()` returns a `PluginInfo` for it with `loaded=False`.
4. `registry.load("toy_language")` imports `cogant_toy_plugin.plugin:ToyLanguagePlugin`, sets `loaded=True`, and caches the loaded class under `_loaded_objects`.
5. COGANT's pipeline can then instantiate the class (optionally calling `initialize(config)`) and dispatch source files by language.

That's it — no editor hooks, no YAML config, no restart. Install the wheel, the plugin is live.

## 6. Takeaways

- Plugin **protocols** live under `cogant.plugins.base` — pick the one that matches your extension (language, trace, normalizer, rule, state-space, process, export, validation, visualization).
- The **registry** is a thin cache around `importlib.metadata`; it's entry-point-driven so anything installed via `pip`/`uv` is automatically discoverable.
- For **tests and notebooks**, the in-process injection pattern shown above lets you exercise the registry without packaging.
- Production plugins should declare themselves in `pyproject.toml` under `[project.entry-points."cogant.plugins"]` so COGANT can load them automatically.